# MeetScribe - Trascrizione riunioni con GPU

Questo notebook esegue la pipeline MeetScribe completa su Google Colab con GPU T4.

**Prima di iniziare:**
1. Vai su `Runtime > Change runtime type > T4 GPU`
2. Avrai bisogno del tuo file audio e del token HuggingFace

## 1. Setup: installa MeetScribe + dipendenze GPU

In [ ]:
import os

REPO_URL = 'https://github.com/andreaceruti/meet-scribe.git'

repo_dir = '/content/meet-scribe'
if not os.path.exists(repo_dir):
    !git clone {REPO_URL} {repo_dir}
else:
    # Aggiorna il repo con le ultime modifiche
    !cd {repo_dir} && git pull
%cd {repo_dir}

# Installa il progetto
!pip install hatchling
!pip install -e .

# Verifica GPU
import torch
print(f"\nGPU disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================
# Robustezza download HuggingFace (impostare PRIMA di ogni download HF).
# Vengono ereditate anche dal subprocess `!python -m meet_scribe.main`.
# ============================================================
# (1) FIX principale dello stallo "0.00B [00:00<?, ?B/s]" su Colab: disabilita il
#     backend Xet, il cui percorso Rust NON rispetta HF_HUB_DOWNLOAD_TIMEOUT e puo'
#     restare appeso a 0 B/s. Senza Xet si usa il download HTTPS classico, che
#     rispetta il timeout e riprende da dove si era fermato. Se Xet non e' attivo,
#     la riga e' innocua.
os.environ["HF_HUB_DISABLE_XET"] = "1"
# (2) Timeout per-lettura socket (default HF = 10s, troppo aggressivo per ~1.6 GB
#     su rete Colab instabile). 30s: fail-fast sullo stallo a 0 B/s, poi resume.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
# (3) hf_transfer NON abilitato di proposito (no resume, no progress bar, richiede
#     un pip install separato, ed e' un no-op deprecato su huggingface_hub>=1.0).


## 2. Configura il token HuggingFace

Serve per scaricare i modelli pyannote (speaker diarization).
Prendi il token da: https://huggingface.co/settings/tokens

In [ ]:
import os
from google.colab import userdata

# Opzione 1: usa Colab Secrets (consigliato)
# Vai su icona chiave nella sidebar > aggiungi 'HF_TOKEN' con il tuo token
try:
    token = userdata.get('HF_TOKEN')
    print('Token caricato da Colab Secrets')
except Exception:
    # Opzione 2: incolla direttamente (meno sicuro)
    token = 'hf_YOUR_TOKEN_HERE'  # <-- sostituisci
    print('Token impostato manualmente')

# Imposta per tutti i componenti (pyannote usa HUGGING_FACE_TOKEN, faster-whisper usa HF_TOKEN)
os.environ['HUGGING_FACE_TOKEN'] = token
os.environ['HF_TOKEN'] = token

## 3. Carica il file audio

Carica il tuo file audio (.m4a, .mp4, .wav, ecc.)

In [ ]:
from google.colab import files
from pathlib import Path

# Upload del file audio
print('Seleziona il file audio da trascrivere...')
uploaded = files.upload()

# Salva nella cartella recordings
recordings_dir = Path('recordings')
recordings_dir.mkdir(exist_ok=True)

for filename, content in uploaded.items():
    audio_path = recordings_dir / filename
    audio_path.write_bytes(content)
    print(f'\nFile salvato: {audio_path}')
    print(f'Dimensione: {len(content) / 1e6:.1f} MB')

## 3b. Modello Whisper + pre-download robusto

Scegli il modello Whisper e **pre-scaricalo prima** di lanciare la pipeline, così un eventuale stallo di rete si vede subito qui (e si ritenta) invece di bloccare il caricamento durante la trascrizione.

- `large-v3-turbo` = migliore su GPU, ma download più grande (~1.6 GB).
- Se si blocca, imposta `WHISPER_MODEL = "medium"` o `"small"` e riesegui la cella.

> La cella disabilita il backend **Xet** (causa n°1 degli stalli a 0 B/s su Colab) e imposta un timeout finito, così i download appesi vengono ritentati invece di restare piantati.

In [ ]:
import os, re, time

# Rete di sicurezza: env var di robustezza attive anche se la cella di setup non
# fosse stata eseguita (qui avviene il primo import di huggingface_hub).
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")
os.environ.pop("HF_HUB_OFFLINE", None)          # il download deve essere ONLINE
os.environ.pop("TRANSFORMERS_OFFLINE", None)

# ------------------------------------------------------------------
# Modello Whisper. "large-v3-turbo" = miglior qualita'/velocita' su GPU, ma su
# Colab T4 e' il download piu' grande (~1.6 GB). SE IL DOWNLOAD/CARICAMENTO SI
# BLOCCA: metti "medium" (~0.5 GB) o "small" (~0.24 GB) qui sotto e riesegui.
# ------------------------------------------------------------------
WHISPER_MODEL = "large-v3-turbo"   # <-- opzioni piu' leggere: "medium" / "small"

CONFIG_PATH = "config.yaml"        # cwd = /content/meet-scribe (root del repo);
                                   # e' lo stesso file che main.py legge.

# ------------------------------------------------------------------
# 1) Patch di config.yaml PRESERVANDO i commenti. Sostituiamo SOLO il valore
#    della riga `model:` (robusto a virgolette doppie/singole/assenti). In
#    config.yaml esiste UNA sola riga `model:` -> l'assert lo garantisce.
# ------------------------------------------------------------------
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg_text = f.read()

MODEL_LINE_RE = re.compile(
    r'''^([ \t]*model:[ \t]*)(['"]?)[^'"#\r\n]*\2([ \t]*(?:\#.*)?)$''',
    re.MULTILINE,
)
n_match = len(MODEL_LINE_RE.findall(cfg_text))
if n_match != 1:
    raise RuntimeError(
        f"Attesa 1 sola riga 'model:' in {CONFIG_PATH}, trovate {n_match}. "
        "Verifica il file prima di procedere (nessuna modifica applicata)."
    )
cfg_text = MODEL_LINE_RE.sub(
    lambda m: f'{m.group(1)}"{WHISPER_MODEL}"{m.group(3)}', cfg_text
)
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    f.write(cfg_text)
print(f'[config] whisper.model = "{WHISPER_MODEL}"  (commenti preservati)')

# ------------------------------------------------------------------
# 2) Pre-download ROBUSTO di Whisper (retry con backoff deterministico; niente
#    retry su 401/403/404 permanenti). download_model risolve il nome breve nel
#    repo HF corretto (turbo -> mobiuslabsgmbh/...) e delega a snapshot_download
#    (resume automatico, rispetta HF_HUB_DOWNLOAD_TIMEOUT / HF_HUB_DISABLE_XET).
#    NB: il caricamento senza hang e' garantito dal codice del repo, che ora
#    costruisce WhisperModel con local_files_only=True (nessun giro di rete al load).
# ------------------------------------------------------------------
from faster_whisper.utils import download_model
from huggingface_hub.utils import HfHubHTTPError

_retry_exc = [HfHubHTTPError, ConnectionError, TimeoutError, OSError]
for _mod in ("httpx", "requests"):
    try:
        _m = __import__(_mod)
        _retry_exc.append(_m.HTTPError if _mod == "httpx" else _m.exceptions.RequestException)
    except Exception:
        pass
RETRY_EXC = tuple(_retry_exc)

whisper_path = None
for tentativo in range(1, 6):
    try:
        whisper_path = download_model(WHISPER_MODEL)
        break
    except RETRY_EXC as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        if code in (401, 403, 404):
            raise                       # errore permanente: non riprovare
        if tentativo == 5:
            break                       # ultimo tentativo: niente sleep finale
        attesa = 2 ** (tentativo - 1)   # 1, 2, 4, 8 s (deterministico)
        print(f"[whisper] tentativo {tentativo}/5 fallito: {type(e).__name__}: {e} "
              f"-- riprovo fra {attesa}s")
        time.sleep(attesa)

if whisper_path is None:
    raise RuntimeError(
        f"Download di '{WHISPER_MODEL}' fallito. Rete Colab instabile o modello "
        'troppo grande. RIMEDIO: imposta WHISPER_MODEL = "medium" (o "small") e riesegui.'
    )
print(f"[whisper] '{WHISPER_MODEL}' pronto in cache: {whisper_path}")
print("[whisper] ora esegui lo step 4 (Esegui MeetScribe).")


### (Opzionale) Ripristina cache modello Whisper

Esegui questa cella **solo se** il modello resta bloccato al caricamento: cancella la cache del modello (inclusi i pezzi corrotti/incompleti lasciati da download interrotti) e lo riscarica pulito. Poi riesegui la cella *"Modello Whisper"* e lo step di trascrizione.

In [ ]:
import os, time, shutil

# Usa lo stesso WHISPER_MODEL della cella "Modello Whisper". Se esegui questa
# cella da sola, decommenta e imposta a mano:
# WHISPER_MODEL = "large-v3-turbo"

# 1) Risolvi il repo HF corretto con la mappa autorevole di faster-whisper
#    ("large-v3-turbo" -> mobiuslabsgmbh/faster-whisper-large-v3-turbo).
try:
    from faster_whisper.utils import _MODELS
    repo_id = _MODELS.get(WHISPER_MODEL, WHISPER_MODEL)
except Exception:
    _FALLBACK = {
        "large-v3-turbo": "mobiuslabsgmbh/faster-whisper-large-v3-turbo",
        "turbo":          "mobiuslabsgmbh/faster-whisper-large-v3-turbo",
        "medium":         "Systran/faster-whisper-medium",
        "small":          "Systran/faster-whisper-small",
        "large-v3":       "Systran/faster-whisper-large-v3",
        "large":          "Systran/faster-whisper-large-v3",
    }
    repo_id = _FALLBACK.get(WHISPER_MODEL, WHISPER_MODEL)

model_dir_name = "models--" + repo_id.replace("/", "--")   # "/" -> "--"

# 2) Root cache HF (stesso ordine di huggingface_hub). Su Colab: /root/.cache/...
if os.environ.get("HF_HUB_CACHE"):
    cache_root = os.environ["HF_HUB_CACHE"]
elif os.environ.get("HF_HOME"):
    cache_root = os.path.join(os.environ["HF_HOME"], "hub")
else:
    cache_root = os.path.expanduser("~/.cache/huggingface/hub")

model_dir = os.path.join(cache_root, model_dir_name)   # PATH ESATTO, niente glob
print(f"[reset] modello  : {WHISPER_MODEL}  ->  {repo_id}")
print(f"[reset] cartella : {model_dir}")

# 3) Elimina l'INTERA cartella del modello (unico reset affidabile: download_model
#    non ha force_download, un blob corrotto ma 'presente' non verrebbe riscaricato).
if os.path.isdir(model_dir):
    shutil.rmtree(model_dir)
    print("[reset] cache rimossa.")
else:
    print("[reset] nessuna cache da rimuovere (cartella inesistente).")

# 4) Ri-scarica pulito, ONLINE, con retry/backoff (niente retry su 401/403/404).
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)

from faster_whisper.utils import download_model
from huggingface_hub.utils import HfHubHTTPError

_retry_exc = [HfHubHTTPError, ConnectionError, TimeoutError, OSError]
for _mod in ("httpx", "requests"):
    try:
        _m = __import__(_mod)
        _retry_exc.append(_m.HTTPError if _mod == "httpx" else _m.exceptions.RequestException)
    except Exception:
        pass
RETRY_EXC = tuple(_retry_exc)

local_path = None
for tentativo in range(1, 6):
    try:
        local_path = download_model(WHISPER_MODEL)
        print(f"[reset] ri-download completato: {local_path}")
        break
    except RETRY_EXC as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        if code in (401, 403, 404):
            raise
        if tentativo == 5:
            break
        attesa = 2 ** (tentativo - 1)
        print(f"[reset] tentativo {tentativo}/5 fallito: {type(e).__name__}: {e} -- riprovo fra {attesa}s")
        time.sleep(attesa)

if local_path is None:
    raise RuntimeError(
        f"Ri-download di '{WHISPER_MODEL}' fallito. Prova WHISPER_MODEL = \"medium\" o \"small\"."
    )

# NB: se il load resta bloccato anche dopo il reset, riavvia il runtime
# (Runtime > Riavvia sessione) per ricaricare huggingface_hub da zero, poi ripeti.
print("[reset] fatto. Riesegui la cella 'Modello Whisper' e poi lo step di trascrizione.")


## 4. Esegui MeetScribe

In [ ]:
# Esegui la pipeline completa
# Usa il file caricato nello step precedente (oppure cambia il path manualmente)
import glob
audio_files = glob.glob('recordings/*')
if audio_files:
    latest = max(audio_files, key=os.path.getmtime)
    print(f"File audio: {latest}")
    !python -m meet_scribe.main --input "{latest}"
else:
    print("Nessun file audio trovato in recordings/. Carica un file nello step 3.")

## 5. Visualizza e scarica i risultati

In [ ]:
from pathlib import Path

# Mostra il file TXT
output_dir = Path('output')
for txt_file in output_dir.glob('*.txt'):
    print(f'=== {txt_file.name} ===')
    print(txt_file.read_text(encoding='utf-8')[:3000])
    print('...(troncato per display)\n')

In [ ]:
# Scarica i file di output
from google.colab import files
from pathlib import Path

output_dir = Path('output')
for f in output_dir.iterdir():
    print(f'Download: {f.name}')
    files.download(str(f))